In [ ]:
import os
import random
from dotenv import load_dotenv
load_dotenv()
from termcolor import colored, cprint

from inspect_ai.log import read_eval_log

In [ ]:
root_dir = os.path.join("..", "outputs", "first_epoch")
eval_dir = "text"

In [ ]:
# For each model, draw `n_sample` correctly and `n_sample` wrongly answered questions
n_sample = 5

results = []
for model_dir in os.listdir(os.path.join(root_dir, eval_dir)):
    model_path = os.path.join(root_dir, eval_dir, model_dir)
    print("Processing: " + colored(model_dir, "cyan"))
  
    # Read evaluation log
    try:
        eval_files = [f for f in os.listdir(model_path) if f.endswith(".eval")]
        log_file = os.path.join(model_path, eval_files[0])
        log = read_eval_log(log_file)
    except Exception as e:
        print(colored(f"\tError reading log for {model_dir}: {e}", "red"))
        continue

    n_models = len(log.stats.model_usage.keys())
    
    # Compute number valid samples
    idx_valid = [i for i, sample in enumerate(log.samples) if sample.score is not None and sample.error is None]
    idx_correct = [i for i, sample in enumerate(log.samples) if i in idx_valid and sample.score.value in [1, True, "C"]]
    idx_wrong = [i for i, sample in enumerate(log.samples) if i in idx_valid and sample.score.value not in [1, True, "C"]]

    idx_sample_correct = random.sample(idx_correct, min(n_sample, len(idx_correct)))
    idx_sample_wrong = random.sample(idx_wrong, min(n_sample, len(idx_wrong)))

    samples_correct = [log.samples[i] for i in idx_sample_correct]
    samples_wrong = [log.samples[i] for i in idx_sample_wrong]
    
    
    results.append({
        "model_name": model_dir.split("_")[0],
        "experiment": root_dir.split("/")[-1],
        "eval_dir": eval_dir,
        "num_samples": num_total_samples,
        "num_valid_samples": num_valid,
        "num_correct_samples": num_correct,
        "accuracy": num_correct / num_total_samples if num_total_samples > 0 else 0,
        "accuracy_valid": num_correct / num_valid if num_valid > 0 else 0,
        "samples_correct": samples_correct,
        "samples_wrong": samples_wrong,
    })